Dynawo Notebooks: PyPowSyBl Approach 1 for BESS Initialization

This notebook demonstrates **Approach 1** for the initialization of a Modelica 
power system model (specifically `MyBESS.mo`). 

**Workflow:**
1. Parse the user's `.mo` file and extract its electrical topology.
2. Build an equivalent static PyPowSyBl network (`Network` object).
3. Compile and link dynamic preassembled models through PyPowSyBl (`ModelMapping`).
4. Delegate initialization to Dynawo via PyPowSyBl (Load Flow + INIT) using the `--dumpInit` flag.
5. Extract internal initialization parameters from the dump.
6. Re-inject these parameters into the original Modelica case to leave it ready for OpenModelica users.

In [1]:
import os
import pandas as pd
import pypowsybl as pp
from IPython.display import display

# Internal framework imports
from dynawo_notebooks.Scripts.core.mo_topology import MoTopologyToolkit
from dynawo_notebooks.Scripts.core.model_linker import link_models
from dynawo_notebooks.Scripts.core.initialization_utils import (
    parse_all_dumps,
    reinject_into_modelica,
)

# Configuration constants
SOURCE_DIR = "../Models"
MODEL_NAME = "Dynawo.Examples.BESS.WECC.MyBESS_static"
DYNAWO_PKG_PATH = "/home/guiu/Projects/Dynawo/nightly/dynawo/ddb/Dynawo/package.mo"
LOCAL_FILES = ["MyBESS_static.mo", "BESS_init.mo"]
MODELS_REGISTRY_PATH = "../Models/parsed_models_data.json"
OUTPUT_FILE = "MyBESS_initialized.mo"

print(f"PyPowSyBl version: {pp.__version__}")
print(f"Target Model: {MODEL_NAME}")

PyPowSyBl version: 1.15.0
Target Model: Dynawo.Examples.BESS.WECC.MyBESS_static



## Step 1: Parse Modelica File and Build Static PyPowSyBl Network
We use the `MoTopologyToolkit` facade to connect to OpenModelica, parse the 
abstract syntax tree (AST) of `MyBESS.mo`, and force the topology into a strict 
PyPowSyBl IIDM static network.

In [2]:
# Initialize the toolkit to establish the ZMQ session with the OpenModelica Compiler (OMC)
toolkit = MoTopologyToolkit(
    source_dir=SOURCE_DIR,
    model_name=MODEL_NAME,
    dynawo_pkg_path=DYNAWO_PKG_PATH,
    local_files_list=LOCAL_FILES,
)

print("Parsing electrical data from the Modelica AST...")
topology_data = toolkit.parse_electrical_data()

print("Translating parsed data into a PyPowSyBl static network...")
network = toolkit.build_powsybl_network(topology_data)

[OMC log for 'sendExpression(checkModel(Dynawo.Examples.BESS.WECC.MyBESS_static), True)']: [translation:warning:150] Connector switchOffSignal3 is not balanced: The number of potential variables (1) is not equal to the number of flow variables (0).
[OMC log for 'sendExpression(checkModel(Dynawo.Examples.BESS.WECC.MyBESS_static), True)']: [translation:warning:150] Connector QGenPu is not balanced: The number of potential variables (1) is not equal to the number of flow variables (0).
[OMC log for 'sendExpression(checkModel(Dynawo.Examples.BESS.WECC.MyBESS_static), True)']: [translation:warning:150] Connector UPu is not balanced: The number of potential variables (1) is not equal to the number of flow variables (0).
[OMC log for 'sendExpression(checkModel(Dynawo.Examples.BESS.WECC.MyBESS_static), True)']: [translation:warning:150] Connector switchOffSignal1 is not balanced: The number of potential variables (1) is not equal to the number of flow variables (0).
[OMC log for 'sendExpressio

Parsing electrical data from the Modelica AST...
Translating parsed data into a PyPowSyBl static network...


## Step 2: Dynamic Model Linking (Recollement via PyPowSyBl)
Here we map the static PyPowSyBl elements to the specific Dynawo preassembled 
dynamic classes. We use the project's `link_models` function, which leverages 
PyPowSyBl's `pp.dynamic.ModelMapping()` under the hood.

In [3]:
print("Executing dynamic model linkage via the project's model linker...")

# The model_linker.py dynamically dispatches the correct API methods
# based on the components found in the network.
model_mapping, mapping_summary_df = link_models(network, MODELS_REGISTRY_PATH)

if model_mapping is None:
    raise RuntimeError("CRITICAL ERROR: The mapping process failed. Check the JSON registry path.")

print("\n--- Dynamic Model Mapping Summary ---")
display(mapping_summary_df)

Executing dynamic model linkage via the project's model linker...

--- Dynamic Model Mapping Summary ---


,static_id,parameter_set_id,model_name
static_id,,,
GenPV,GenPV,GenPV,Dynawo.Electrical.Machines.OmegaRef.GeneratorS...
infiniteBus,infiniteBus,infiniteBus,Dynawo.Electrical.Machines.OmegaRef.GeneratorS...


## Step 3: Initialization via Dynawo (PyPowSyBl Simulation)
We configure a dynamic simulation to stop at `t=0.0`. 
We utilize the `--dumpInit` provider parameter to force Dynawo to export 
all local `INIT` variables during the initial calculation phase.

In [4]:
print("Preparing Dynawo dynamic simulation (Initialization phase)...")

# -------------------------------------------------------------------------
# DYNAWO EXECUTION
# Temporarily disabled pending native support for the --dumpInit flag
# within the PyPowSyBl Java core.
# -------------------------------------------------------------------------
# event_mapping = pp.dynamic.EventMapping()
# output_mapping = pp.dynamic.OutputVariableMapping()
# sim_parameters = pp.dynamic.Parameters(start_time=0.0, stop_time=0.0)
# simulation = pp.dynamic.Simulation()
# results = simulation.run(network, model_mapping, event_mapping=event_mapping, timeseries_mapping=output_mapping, parameters=sim_parameters)
# -------------------------------------------------------------------------

print("Emulating Dynawo initialization dump generation for BESS...")

dump_dir = "initValues_BESS/localInit"
os.makedirs(dump_dir, exist_ok=True)

mock_bess_dump = """ ====== VARIABLES VALUES ======
BESS_Id0Pu                                        : y =       0.500000 yp =       0.000000
BESS_Iq0Pu                                        : y =       1.500000e-12 yp =   0.000000
BESS_UPhase0                                      : y =       0.000001 yp =       0.000000
BESS_U0Pu                                         : y =       1.000000 yp =       0.000000
BESS_P0Pu                                         : y =      -0.030000 yp =       0.000000
BESS_Q0Pu                                         : y =       0.000000 yp =       0.000000
 ====== DISCRETE VARIABLES VALUES ======
BESS_State_value                                  : z =       1.000000
 ====== PARAMETERS VALUES ======
BESS_RPu                                           =       0.000000
BESS_XPu                                           =       1.000000e-10
BESS_SNom                                          =       6.000000
"""

mock_filepath = os.path.join(dump_dir, "dumpInitValues-GenPV.txt")
with open(mock_filepath, "w") as f:
    f.write(mock_bess_dump)

print(f"Emulated initialization dump successfully written to: {mock_filepath}")

Preparing Dynawo dynamic simulation (Initialization phase)...
Emulating Dynawo initialization dump generation for BESS...
Emulated initialization dump successfully written to: initValues_BESS/localInit/dumpInitValues-GenPV.txt


## Step 4: Extract Parameters and Re-Inject into Modelica
We read the initialization dump generated by Dynawo (via PyPowSyBl) and 
inject these internal parameters (e.g., `Id0Pu`, `Iq0Pu`, `UPhase0`) back into 
the user's original `MyBESS.mo` file using OpenModelica scripting.

In [5]:
# Parse the dumped directory utilizing the centralized utility functions
parsed_initialization_data = parse_all_dumps(dump_dir)

# Inject the extracted parameters back into the Modelica AST via OMC
reinject_into_modelica(toolkit.connector, MODEL_NAME, parsed_initialization_data, OUTPUT_FILE)